In [4]:
import sys
import torch
import torch_geometric
import torch_scatter
import torch_sparse
import torch_cluster
import torch_spline_conv
import rdkit
import pandas as pd
import numpy as np
import scipy
import sklearn
import xgboost
import tqdm
import matplotlib
import seaborn
import streamlit
import requests

print("Success: All packages imported successfully!")
print(f"-> Using Python: {sys.executable}")
print(f"-> GPU Available: {torch.cuda.is_available()}")

Success: All packages imported successfully!
-> Using Python: /home/satyam-rana/miniconda3/envs/pytorch_env/bin/python
-> GPU Available: True


In [6]:
import os, json
import numpy as np

# Directory helpers
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def save_json(data, path: str):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(data, f)

def load_json(path: str):
    with open(path, "r") as f:
        return json.load(f)

# STITCH ID helpers 
def normalize_stitch_id(sid: str) -> str:
    """Normalize any STITCH ID to uppercase stripped string."""
    return str(sid).strip().upper()

def stitch_to_pubchem_cid(sid: str) -> int:
    """Extract numeric PubChem CID from a STITCH ID (CIDs/CID1 prefix)."""
    sid = normalize_stitch_id(sid)
    if sid.startswith("CID1"):
        return int(sid[4:])
    if sid.startswith("CIDS"):
        return int(sid[4:])
    numeric = "".join(c for c in sid if c.isdigit())
    return int(numeric) if numeric else -1

def compute_pos_weight(labels: np.ndarray) -> float:
    """Return neg/pos ratio for BCEWithLogitsLoss pos_weight."""
    pos = labels.sum()
    neg = len(labels) - pos
    return neg / max(pos, 1)

# Directory setup 
DATA_DIR      = "data"
RAW_DIR       = os.path.join(DATA_DIR, "raw")
PROC_DIR      = os.path.join(DATA_DIR, "processed")
MAP_DIR       = os.path.join(DATA_DIR, "mappings")
MODELS_DIR    = "models"

for d in [RAW_DIR, PROC_DIR, MAP_DIR, MODELS_DIR]:
    ensure_dir(d)

print("Directories ready:", DATA_DIR, MODELS_DIR)

Directories ready: data models
